# 혜진 v8: 가이드 기반 상호작용·학습 내 지문 점수·쌍별 감별
CANCER_GENOME_GUIDE.md에서 제안한 유전자/위치/조합을 후보 특성으로 사용합니다. 문서의 진단 확률이나 test 암종 수를 정답으로 간주하지 않습니다. 변이 수는 이 패널의 변이 유전자 수이며 mutations/Mb 단위 TMB가 아닙니다. 관측 WT는 해당 데이터의 기록일 뿐 생물학적 정상이나 측정 완전성을 보증하지 않습니다.

원본 EDA, 변수 생성·선별·학습·검증·피클·CSV·문서 생성을 노트북에 포함했습니다. 이전 피클/프로젝트 Python 파일은 필수가 아니며 캐시가 없으면 다시 학습합니다. 원본 data/와 requirements-v8.txt 환경은 필요합니다. 문서 자체가 전체 train을 분석한 자료이므로 설계에 따른 선택 편향이 있습니다. 기존 홀드아웃/재분할은 독립 새 환자의 최종 평가가 아닙니다.

추가 대조 실험: 초기 8개 모델 결과를 본 뒤, 기존 v6 로지스틱/LightGBM 구성 모델의 선택 변수 2,048/4,096개를 그대로 보존하고 가이드/지문 변수만 덧붙인 4개 설정을 추가했습니다. 총 12개 신규 모델이며 적응 탐색의 선택 편향이 있습니다. 추가 변수는 기준 입력과 완전히 같은 열을 제외합니다. 입력 수는 기준보다 약간 늘어납니다. Run All은 12개를 처음부터 평가합니다.

In [1]:
import os,sys,re,json,pickle,cloudpickle,hashlib,platform,time,copy,warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.special import softmax
from sklearn.base import BaseEstimator,TransformerMixin,clone
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_selection import chi2,f_classif
from sklearn.preprocessing import StandardScaler,MaxAbsScaler
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import f1_score,accuracy_score,classification_report,confusion_matrix
from lightgbm import LGBMClassifier
from threadpoolctl import threadpool_limits
from IPython.display import display,FileLink
threadpool_limits(2)
SEED=42
roots=[Path.cwd(),*Path.cwd().parents]
REPO=next((p for p in roots if all((p/'data'/f).is_file() for f in ['train.csv','test.csv','sample_submission.csv'])),None)
if REPO is None:raise FileNotFoundError('공用 data/에 train.csv, test.csv, sample_submission.csv가 필요합니다.')
DATA=REPO/'data';WORK=REPO/'models'/'혜진' if (REPO/'models'/'혜진').is_dir() else REPO
OUT=WORK/'artifacts'/'v8';OUT.mkdir(parents=True,exist_ok=True)
SUBMIT=REPO/'submission';SUBMIT.mkdir(exist_ok=True)
os.environ['MPLCONFIGDIR']=str(OUT/'.mplconfig')
train=pd.read_csv(DATA/'train.csv');test=pd.read_csv(DATA/'test.csv');sample=pd.read_csv(DATA/'sample_submission.csv')
genes=[c for c in train if c not in ['ID','SUBCLASS']]
assert list(test)==['ID']+genes and sample.ID.equals(test.ID)
assert train.ID.is_unique and test.ID.is_unique
y=train.SUBCLASS.to_numpy();labels=np.sort(np.unique(y));yi=np.searchsorted(labels,y);NC=len(labels)
X=train[genes];groups=pd.util.hash_pandas_object(X,index=False).to_numpy()
dev,hold=next(StratifiedGroupKFold(5,shuffle=True,random_state=SEED).split(X,y,groups))
folds=list(StratifiedGroupKFold(3,shuffle=True,random_state=SEED+1).split(X.iloc[dev],y[dev],groups[dev]))
assert not set(groups[dev]).intersection(groups[hold])
for a,b in folds:assert not set(groups[dev[a]]).intersection(groups[dev[b]])
pd.DataFrame({'ID':train.ID,'split':np.where(np.isin(np.arange(len(train)),hold),'holdout','development'),'group':groups.astype(str)}).to_csv(OUT/'split_manifest.csv',index=False)
print('DATA',DATA,'OUTPUT',OUT,'Train/test',train.shape,test.shape,'Dev/holdout',len(dev),len(hold))

from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier


DATA /Users/mac/project/cancer_project/data OUTPUT /Users/mac/project/cancer_project/artifacts/v8 Train/test (6201, 4386) (2546, 4385) Dev/holdout 4960 1241


In [2]:
# Descriptive EDA only: no test-derived vocabularies or fitted preprocessing.
burden_train=(train[genes].notna()&train[genes].ne('WT')).sum(axis=1)
burden_test=(test[genes].notna()&test[genes].ne('WT')).sum(axis=1)
eda=pd.DataFrame({'train':burden_train.describe(),'test':burden_test.describe()})
eda.to_csv(OUT/'eda_mutated_gene_burden.csv')
byclass=pd.DataFrame({'SUBCLASS':y,'mutated_genes':burden_train}).groupby('SUBCLASS').mutated_genes.agg(['count','mean','median','max'])
byclass.to_csv(OUT/'eda_by_class.csv')
missing=pd.DataFrame({'train_missing':train[genes].isna().sum(),'test_missing':test[genes].isna().sum()})
missing.to_csv(OUT/'eda_missing_by_gene.csv')
print('4384 original gene columns; mutation burden is the number of non-WT non-missing gene cells, not the token count.')
display(eda);display(byclass)


4384 original gene columns; mutation burden is the number of non-WT non-missing gene cells, not the token count.


,train,test
count,6201.000000,2546.000000
mean,35.299629,78.134328
std,100.776379,213.964535
min,0.000000,0.000000
25%,7.000000,11.000000
50%,14.000000,28.000000
75%,27.000000,52.000000
max,2393.000000,2784.000000


,count,mean,median,max
SUBCLASS,,,,
ACC,72,39.430556,24.0,396
BLCA,104,70.490385,54.0,382
BRCA,786,17.437659,9.0,853
CESC,155,44.129032,20.0,480
COAD,223,73.121076,23.0,1631
DLBC,38,67.421053,30.0,704
GBMLGG,461,12.535792,11.0,121
HNSC,223,40.569507,31.0,386
KIPAN,515,16.042718,15.0,246


In [3]:
GROUPS = {
 'p53_cellcycle': 'TP53 RB1 CDKN2A CDKN2B CDK4 CDK6 CCND1 ATM CHEK2'.split(),
 'rtk_ras_mapk': 'EGFR ERBB2 ERBB3 KRAS NRAS HRAS BRAF NF1 MAP2K1 MAP2K2 KIT PDGFRA FGFR2 FGFR3'.split(),
 'pi3k_mtor': 'PIK3CA PIK3R1 PTEN AKT1 MTOR TSC1 TSC2 STK11'.split(),
 'dna_repair': 'BRCA1 BRCA2 PALB2 ATM ATR CHEK2 MLH1 MSH2 MSH6 PMS2 POLE POLD1'.split(),
 'chromatin': 'ARID1A ARID1B ARID2 PBRM1 SMARCA4 BAP1 SETD2 KMT2A KMT2C KMT2D ATRX'.split(),
 'wnt': 'APC CTNNB1 AXIN1 AXIN2 RNF43'.split(),
 'metabolism': 'IDH1 IDH2 SDHB SDHC SDHD FH VHL'.split(),
}

def kind(token):
 if 'fs' in token.lower(): return 'frameshift'
 if '*' in token: return 'stop'
 if 'del' in token.lower(): return 'deletion'
 if 'ins' in token.lower(): return 'insertion'
 m = re.fullmatch(r'([A-Z])(\d+)([A-Z])', token)
 if m: return 'synonymous' if m[1] == m[3] else 'substitution'
 return 'other'

class MutationFeatures(BaseEstimator, TransformerMixin):
 def __init__(self, mode='full', min_df=3):
  self.mode = mode
  self.min_df = min_df

 def dictionaries(self, X):
  genes = list(X.columns)
  for row in X.itertuples(index=False, name=None):
   d = {}; counts = {}; burden = 0; events = 0; positions = []
   changed = set()
   for gene, value in zip(genes, row):
    if value is None or str(value) in ('nan', '', '<NA>'):
     d['missing:' + gene] = 1; continue
    if value == 'WT': continue
    changed.add(gene); burden += 1; d['gene:' + gene] = 1
    if self.mode == 'binary': continue
    tokens = str(value).split()
    for raw_token in tokens:
     token = re.sub(r'^([A-Z]\d+)X$', r'\1*', raw_token)
     events += 1; typ = kind(token); counts[typ] = counts.get(typ, 0) + 1
     d['variant:' + gene + ':' + token] = 1
     d['type:' + gene + ':' + typ] = 1
     m = re.search(r'\d+', token)
     if m:
      position = int(m[0]); positions.append(position)
      d['position:' + gene + ':' + str(position)] = 1
   if self.mode == 'full':
    d['numeric:log_mutated_genes'] = np.log1p(burden)
    d['numeric:log_events'] = np.log1p(events)
    d['numeric:multi_event_excess'] = np.log1p(max(0, events-burden))
    for typ in ['frameshift','stop','deletion','insertion','synonymous','substitution','other']:
     d['numeric:log_' + typ] = np.log1p(counts.get(typ,0))
     d['numeric:fraction_' + typ] = counts.get(typ,0)/max(events,1)
    d['numeric:log_mean_position'] = np.log1p(np.mean(positions)) if positions else 0.
    for name, members in GROUPS.items():
     d['module:' + name] = np.log1p(len(changed.intersection(members)))
   yield d

 def fit(self, X, y=None):
  self.genes_ = list(X.columns)
  self.vectorizer_ = DictVectorizer(dtype=np.float32)
  matrix = self.vectorizer_.fit_transform(self.dictionaries(X))
  names = self.vectorizer_.get_feature_names_out()
  self.keep_ = (np.asarray((matrix != 0).sum(axis=0)).ravel() >= self.min_df) | np.array([s.startswith(('numeric:', 'module:')) for s in names])
  self.feature_names_ = names[self.keep_]
  return self

 def transform(self, X):
  missing = set(self.genes_) - set(X.columns)
  if missing: raise ValueError('Missing gene columns: ' + str(sorted(missing)[:10]))
  return sparse.csr_matrix(self.vectorizer_.transform(self.dictionaries(X[self.genes_]))[:, self.keep_])

 def get_feature_names_out(self, input_features=None): return self.feature_names_
def canon(t):
 t=t.strip().removeprefix('p.');t=re.sub(r'(?<=\d)X$','*',t)
 return re.sub(r'^([A-Z]\d+)(?:[A-Z*]+)?fs.*$',r'\1fs',t)
def aa_change(t):
 m=re.fullmatch(r'([A-Z])\d+([A-Z*])',t)
 if m:return m[1]+'>'+m[2]
 m=re.match(r'([A-Z])\d+',t)
 if m:
  for suffix in ['fs','del','ins']:
   if suffix in t:return m[1]+'>'+suffix
 return None

def parse_events(frame):
 names=list(frame.columns);result=[]
 for row in frame.itertuples(index=False,name=None):
  result.append([(g,tuple(sorted(set(canon(t) for t in v.split())))) for g,v in zip(names,row) if isinstance(v,str) and v!='WT'])
 return result

class EvidenceEncoder:
 def __init__(self,unknown=1.,silent=1.,support=2):self.unknown=unknown;self.silent=silent;self.support=support
 def fit(self,events):
  self.token_df={};self.change_df={}
  if self.unknown<1:
   tokens=Counter();changes=Counter()
   for row in events:
    for g,ts in row:
     tokens.update((g,t) for t in ts)
     changes.update(set((g,a) for t in ts if (a:=aa_change(t)) is not None))
   self.token_df={k:v for k,v in tokens.items() if v>=self.support}
   self.change_df={k:v for k,v in changes.items() if v>=self.support}
  return self
 def transform(self,events,training=False,exclude_events=None):
  result=[]
  for row_number,row in enumerate(events):
   own=exclude_events[row_number] if exclude_events is not None else (row if training else [])
   own_tokens={(gg,tt) for gg,tts in own for tt in tts}
   own_changes={(gg,ac) for gg,tts in own for tt in tts if (ac:=aa_change(tt)) is not None}
   d={};gene_weights={};type_sums=Counter();non_syn=0.;trunc=0.;multikind=0
   for g,ts in row:
    types={};gene=0.;known_gene=False
    for t in ts:
     typ=kind(t);a=aa_change(t)
     known=self.token_df.get((g,t),0)-int((g,t) in own_tokens)>=self.support
     known_aa=a is not None and self.change_df.get((g,a),0)-int((g,a) in own_changes)>=self.support
     if typ=='synonymous':w=self.silent
     elif known or typ in ['stop','frameshift','deletion','insertion']:w=1.
     elif known_aa:w=max(self.unknown,.75)
     else:w=self.unknown
     if w<=0:continue
     gene=max(gene,w);types[typ]=max(types.get(typ,0),w)
     d['variant:'+g+':'+t]=max(d.get('variant:'+g+':'+t,0),w)
     m=re.search(r'\d+',t)
     if m:d['position:'+g+':'+m[0]]=max(d.get('position:'+g+':'+m[0],0),w)
     if a:d['aa:'+g+':'+a]=max(d.get('aa:'+g+':'+a,0),w)
     if known and typ!='synonymous':known_gene=True
    if gene<=0:continue
    d['gene:'+g]=gene;gene_weights[g]=gene
    for typ,w in types.items():d['type:'+g+':'+typ]=w;type_sums[typ]+=w
    if known_gene:d['supported:'+g]=1.
    non_syn+=max([w for k,w in types.items() if k!='synonymous'] or [0.])
    trunc+=max([w for k,w in types.items() if k in ['stop','frameshift','deletion','insertion']] or [0.])
    multikind+=len(types)>1
   n=len(row)
   d['numeric:log_raw_genes']=np.log1p(n)
   d['numeric:log_effective_genes']=np.log1p(sum(gene_weights.values()))
   d['numeric:log_nonsyn_genes']=np.log1p(non_syn)
   d['numeric:log_trunc_genes']=np.log1p(trunc)
   d['numeric:log_multikind_genes']=np.log1p(multikind)
   for typ in ['synonymous','substitution','stop','frameshift','deletion','insertion','other']:
    d['numeric:gene_fraction_'+typ]=type_sums[typ]/max(n,1)
   for name,gs in GROUPS.items():d['module:'+name]=np.log1p(sum(gene_weights.get(g,0) for g in gs))
   result.append(d)
  return result

events=parse_events(X)
legacy_records=list(MutationFeatures(mode='full').dictionaries(X))
assert canon('R213X')=='R213*' and canon('K16Rfs')=='K16fs'
assert aa_change('R132H')==aa_change('R140H')=='R>H'
print('Parsed rows:',len(events),'Normalization examples passed')


Parsed rows: 6201 Normalization examples passed


In [4]:
def relevance_scores(a,target,method):
 base=method.removesuffix('_full')
 if base=='embedded':
  scaled=csr32(MaxAbsScaler().fit_transform(a))
  fitted=OneVsRestClassifier(LogisticRegression(solver='liblinear',l1_ratio=1,C=.2,class_weight='balanced',max_iter=2000),n_jobs=1).fit(scaled,target)
  return np.max(np.abs(np.vstack([e.coef_[0] for e in fitted.estimators_])),axis=0)
 return (chi2(a,target) if base=='chi2' else f_classif(a,target))[0]

def csr32(a):
 b=sparse.csr_matrix(a,dtype=np.float32)
 if b.nnz>=np.iinfo(np.int32).max:raise ValueError('Sparse matrix too large for tree index type')
 b.indices=b.indices.astype(np.int32,copy=False);b.indptr=b.indptr.astype(np.int32,copy=False)
 return b

class Space:
 def fit(self,rs):
  self.vec=DictVectorizer(dtype=np.float32);a=self.vec.fit_transform(rs)
  names=self.vec.get_feature_names_out();df=np.asarray((a!=0).sum(axis=0)).ravel()
  keep=(df>=3)|np.array([n.startswith(('numeric:','module:')) for n in names])
  self.raw=np.flatnonzero(keep);self.names=names[keep]
  return csr32(a[:,self.raw])
 def transform(self,rs):return csr32(self.vec.transform(rs)[:,self.raw])
 def compact_vec(self,indices):
  v=copy.deepcopy(self.vec);mask=np.zeros(len(v.feature_names_),bool);mask[self.raw[np.sort(indices)]]=True
  v.restrict(mask);return v

class Selector:
 def __init__(self,a,target,names):
  self.a=a;self.y=target;self.names=names;self.cache={};self.corr={};self.scores={};self.audit=[];self.examples=[]
 def order(self,method,threshold):
  key=(method,threshold)
  if key in self.cache:return self.cache[key]
  if method not in self.scores:
   with warnings.catch_warnings():
    warnings.simplefilter('ignore',RuntimeWarning);score=relevance_scores(self.a,self.y,method)
   score=np.nan_to_num(score,nan=0,posinf=np.finfo(np.float32).max,neginf=0)
   if not method.endswith('_full'):score[np.array([n.startswith('numeric:') for n in self.names])]=0
   self.scores[method]=score
  score=self.scores[method]
  if method not in self.corr:
   pool=np.argsort(-score,kind='stable')[:min(4096,int((score>0).sum()))]
   z=self.a[:,pool];mean=np.asarray(z.mean(axis=0)).ravel();var=np.asarray(z.multiply(z).mean(axis=0)).ravel()-mean**2
   sd=np.sqrt(np.maximum(var,1e-12));corr=(z.T@z).toarray()/z.shape[0]-np.outer(mean,mean)
   corr/=np.outer(sd,sd);np.clip(corr,-1,1,out=corr)
   self.corr[method]=(pool,corr)
  pool,corr=self.corr[method];chosen=[]
  for i in range(len(pool)):
   if chosen:
    correlations=np.abs(corr[i,chosen]);j=int(np.argmax(correlations))
    if correlations[j]>=threshold:
     if len(self.examples)<100:self.examples.append({'method':method,'threshold':threshold,'kept':self.names[pool[chosen[j]]],'discarded':self.names[pool[i]],'correlation':float(corr[i,chosen[j]])})
     continue
   chosen.append(i)
   if len(chosen)>=2048:break
  order=pool[chosen];self.cache[key]=order
  self.audit.append({'method':method,'threshold':threshold,'association_pool':len(pool),'remaining_up_to_2048':len(order)})
  return order
 def select(self,method,threshold,k):return np.sort(self.order(method,threshold)[:k])

class CompactPredictor:
 def __init__(self,space,cols,est):
  self.vec=space.compact_vec(cols);self.names=self.vec.get_feature_names_out();self.est=est
  self.direct_inputs=len(self.names);self.raw_inputs=len(self.names);self.kind='compact'
 def predict(self,rs):return self.est.predict_proba(csr32(self.vec.transform(rs)))

def background_frame(frame,training_frame,seed,count):
 rng=np.random.default_rng(seed);out=frame.copy();arr=out.to_numpy(copy=True)
 rate=(training_frame.notna()&training_frame.ne('WT')).mean().to_numpy();prob=rate/rate.sum();idx=np.flatnonzero(prob>0)
 for r in range(len(arr)):
  n=min(len(idx),int(rng.poisson(count)))
  for c in rng.choice(len(prob),size=n,replace=False,p=prob):
   token='A1A' if rng.random()<.5 else 'A1V'
   arr[r,c]=token if not isinstance(arr[r,c],str) or arr[r,c]=='WT' else arr[r,c]+' '+token
 return pd.DataFrame(arr,columns=frame.columns,index=frame.index)
class ColumnPost:
 def __init__(self,mode='none'):self.mode=mode
 def fit(self,a,names):
  self.raw_names=np.array(names);self.names=np.array(names)
  if self.mode=='maxabs':self.scaler=MaxAbsScaler().fit(a)
  if self.mode=='block':
   self.binary=np.flatnonzero([not n.startswith(('numeric:','module:')) for n in names])
   self.summary=np.flatnonzero([n.startswith(('numeric:','module:')) for n in names])
   self.tf=TfidfTransformer(norm='l2',use_idf=True).fit(a[:,self.binary])
   self.scaler=MaxAbsScaler().fit(a[:,self.summary])
   self.names=np.concatenate([self.raw_names[self.binary],self.raw_names[self.summary]])
  return self
 def transform(self,a):
  if self.mode=='maxabs':return csr32(self.scaler.transform(a))
  if self.mode=='block':return csr32(sparse.hstack([self.tf.transform(a[:,self.binary]),self.scaler.transform(a[:,self.summary])],format='csr'))
  return csr32(a)

class Member:
 def __init__(self,name,view,encoder,space,cols,post,est):
  self.name=name;self.view=view;self.encoder=encoder;self.vec=space.compact_vec(cols);self.post=post;self.est=est
  self.raw_names=self.vec.get_feature_names_out();self.names=post.names
 def predict_rows(self,rs):return self.est.predict_proba(self.post.transform(csr32(self.vec.transform(rs))))

VIEWS={'raw':(1.,1.),'syn_down':(1.,.25),'gated':(.25,1.),'gated_syn_down':(.25,.25),'gated_nonsyn':(.25,0.)}
CONFIGS={}
for view in VIEWS:
 for k in [1024,2048]:
  for c in [.1,.5]:CONFIGS[f'{view}_lr_k{k}_C{c}']=dict(view=view,algo='lr',k=k,C=c,post='none',method='chi2_full')
  CONFIGS[f'{view}_lgb_k{k}']=dict(view=view,algo='lgb',k=k,post='none',method='f_classif_full')
for view in ['raw','gated_syn_down']:
 for depth in [3,5]:CONFIGS[f'{view}_xgb_d{depth}']=dict(view=view,algo='xgb',k=2048,depth=depth,post='none',method='f_classif_full')
 for alpha in [1,10]:CONFIGS[f'{view}_mlp_a{alpha}']=dict(view=view,algo='mlp',k=1024,alpha=alpha,post='maxabs',method='chi2_full')
 for c in [1,4]:CONFIGS[f'{view}_blocklr_C{c}']=dict(view=view,algo='lr',k=2048,C=c,post='block',method='chi2_full')
 CONFIGS[f'{view}_extra']=dict(view=view,algo='extra',k=1024,post='none',method='f_classif_full')
(OUT/'configurations.json').write_text(json.dumps(CONFIGS,indent=2))

def estimator(cfg):
 algo=cfg['algo']
 if algo=='lr':return LogisticRegression(C=cfg['C'],class_weight='balanced',max_iter=2000)
 if algo=='lgb':return LGBMClassifier(n_estimators=300,learning_rate=.04,num_leaves=15,min_child_samples=20,reg_lambda=10,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=SEED)
 if algo=='xgb':return XGBClassifier(n_estimators=300,max_depth=cfg['depth'],learning_rate=.04,min_child_weight=3,reg_lambda=10,subsample=.85,colsample_bytree=.85,max_bin=63,tree_method='hist',objective='multi:softprob',eval_metric='mlogloss',n_jobs=2,random_state=SEED)
 if algo=='mlp':return MLPClassifier(hidden_layer_sizes=(128,),activation='relu',alpha=cfg['alpha'],learning_rate_init=.001,batch_size=128,max_iter=250,tol=1e-4,n_iter_no_change=15,early_stopping=False,random_state=SEED)
 return ExtraTreesClassifier(n_estimators=250,max_depth=20,min_samples_leaf=3,max_features='sqrt',class_weight='balanced',n_jobs=2,random_state=SEED)

def prepare_view(idx,view):
 ev=[events[i] for i in idx];enc=EvidenceEncoder(*VIEWS[view]).fit(ev)
 rs=enc.transform(ev,training=True);space=Space();a=space.fit(rs);selector=Selector(a,yi[idx],space.names)
 return enc,space,a,selector

def train_new(name,idx,cache):
 cfg=CONFIGS[name];view=cfg['view']
 if view not in cache:cache[view]=prepare_view(idx,view)
 enc,space,a,selector=cache[view];cols=selector.select(cfg['method'],.98,cfg['k'])
 post=ColumnPost(cfg['post']).fit(a[:,cols],space.names[cols]);tx=post.transform(a[:,cols]);target=yi[idx]
 row_weights=np.ones(len(idx),dtype=float)
 if cfg.get('augment',False):
  if '_aug_events' not in cache:
   cache['_aug_events']=parse_events(rare_background(X.iloc[idx],[events[i] for i in idx],SEED+777,count=10))
  aug_rs=enc.transform(cache['_aug_events'],exclude_events=[events[i] for i in idx])
  aug_x=post.transform(space.transform(aug_rs)[:,cols]);tx=csr32(sparse.vstack([tx,aug_x],format='csr'))
  target=np.concatenate([target,target]);row_weights=np.concatenate([row_weights,.5*row_weights])
 est=estimator(cfg)
 if cfg['algo'] in ['xgb','mlp']:est.fit(tx,target,sample_weight=compute_sample_weight('balanced',target)*row_weights)
 elif cfg.get('augment',False):est.fit(tx,target,sample_weight=row_weights)
 else:est.fit(tx,target)
 return Member(name,view,enc,space,cols,post,est)

LEGACY={
 'legacy_lr':dict(k=2048,method='chi2',algo='lr'),
 'legacy_lgb':dict(k=2048,method='f_classif_full',algo='lgb'),
 'legacy_extra':dict(k=1024,method='f_classif_full',algo='extra')}
def train_legacy(idx):
 space=Space();a=space.fit([legacy_records[i] for i in idx]);sel=Selector(a,yi[idx],space.names);result=[]
 for name,cfg in LEGACY.items():
  cols=sel.select(cfg['method'],.95,cfg['k'])
  if cfg['algo']=='lr':est=LogisticRegression(C=.1,class_weight='balanced',max_iter=2000)
  elif cfg['algo']=='lgb':est=LGBMClassifier(n_estimators=250,learning_rate=.05,num_leaves=15,min_child_samples=20,reg_lambda=10,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=SEED)
  else:est=ExtraTreesClassifier(n_estimators=250,max_depth=20,min_samples_leaf=3,max_features='sqrt',class_weight='balanced',n_jobs=2,random_state=SEED)
  est.fit(a[:,cols],yi[idx]);post=ColumnPost().fit(a[:,cols],space.names[cols]);result.append(Member(name,'legacy',None,space,cols,post,est))
 return result

def duplicate_frame(frame):
 arr=frame.to_numpy(copy=True)
 for i,row in enumerate(arr):
  for j,v in enumerate(row):
   if isinstance(v,str) and v!='WT':arr[i,j]=v+' '+v
 return pd.DataFrame(arr,columns=frame.columns,index=frame.index)

def rare_background(frame,tr_events,seed,count=20):
 rng=np.random.default_rng(seed);df=Counter((g,t) for row in tr_events for g,ts in row for t in ts)
 syn=[];sub=[]
 for (g,t),n in df.items():
  if n>2:continue
  if kind(t)=='synonymous':syn.append((g,t))
  elif kind(t)=='substitution':sub.append((g,t))
 arr=frame.to_numpy(copy=True);lookup={g:i for i,g in enumerate(frame.columns)}
 for row in arr:
  for _ in range(int(rng.poisson(count))):
   pool=syn if rng.random()<.5 else sub
   g,t=pool[int(rng.integers(len(pool)))];j=lookup[g]
   row[j]=t if not isinstance(row[j],str) or row[j]=='WT' else row[j]+' '+t
 return pd.DataFrame(arr,columns=frame.columns,index=frame.index)

def eval_views(idx,tr_idx,seed):
 clean=X.iloc[idx];noise=rare_background(clean,[events[i] for i in tr_idx],seed);dup=duplicate_frame(clean)
 frames=[clean,noise,dup]
 return frames,[parse_events(f) for f in frames],[list(MutationFeatures(mode='full').dictionaries(f)) for f in frames]

def predict_members(members,weights,frame):
 ev=None;old=None;cache={};p=None
 for m,w in zip(members,weights):
  if m.view=='legacy':
   if old is None:old=list(MutationFeatures(mode='full').dictionaries(frame[genes]))
   rs=old
  else:
   if ev is None:ev=parse_events(frame[genes])
   if m.view not in cache:cache[m.view]=m.encoder.transform(ev)
   rs=cache[m.view]
  q=m.predict_rows(rs);p=q*w if p is None else p+q*w
 return p
print('Loaded frozen historical definitions; no historical search is rerun.')

# CV-guided targeted refinement: regularization, support threshold, and rare-variant augmentation.
VIEWS['gated_syn_down_s3']=(.25,.25,3)
CONFIGS.update({'raw_mlp_a0.01': {'view': 'raw', 'algo': 'mlp', 'k': 1024, 'alpha': 0.01, 'post': 'maxabs', 'method': 'chi2_full'}, 'raw_mlp_a0.1': {'view': 'raw', 'algo': 'mlp', 'k': 1024, 'alpha': 0.1, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_mlp_a0.01': {'view': 'gated_syn_down', 'algo': 'mlp', 'k': 1024, 'alpha': 0.01, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_mlp_a0.1': {'view': 'gated_syn_down', 'algo': 'mlp', 'k': 1024, 'alpha': 0.1, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k1024_C0.1': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 1024, 'C': 0.1, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k1024_C0.5': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 1024, 'C': 0.5, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lgb_k1024': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 1024, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_lr_k2048_C0.1': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 2048, 'C': 0.1, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k2048_C0.5': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 2048, 'C': 0.5, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lgb_k2048': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_xgb_d3': {'view': 'gated_syn_down_s3', 'algo': 'xgb', 'k': 2048, 'depth': 3, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_xgb_d5': {'view': 'gated_syn_down_s3', 'algo': 'xgb', 'k': 2048, 'depth': 5, 'post': 'none', 'method': 'f_classif_full'}, 'raw_aug_lgb': {'view': 'raw', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full', 'augment': True}, 'raw_aug_xgb_d5': {'view': 'raw', 'algo': 'xgb', 'k': 2048, 'depth': 5, 'post': 'none', 'method': 'f_classif_full', 'augment': True}, 'gated_s3_aug_lgb': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full', 'augment': True}})
(OUT/'configurations.json').write_text(json.dumps(CONFIGS,indent=2))
print('Historical comparator ready.')


Novel settings: 44 Views: {'raw': (1.0, 1.0), 'syn_down': (1.0, 0.25), 'gated': (0.25, 1.0), 'gated_syn_down': (0.25, 0.25), 'gated_nonsyn': (0.25, 0.0)}
Total new configurations after refinement: 59


In [5]:
# Freeze the previous version; do not repeat its 59-setting search.
BASE_CONFIGS=copy.deepcopy(CONFIGS)
BASE_PLAN=[('v4_reference',.25),('gated_s3_aug_lgb',.1875),('raw_aug_xgb_d5',.1875),('raw_lr_k2048_C0.1',.1875),('gated_syn_down_mlp_a1',.1875)]

def fit_base(idx):
 cache={};ms=[];ws=[]
 for name,w in BASE_PLAN:
  if name=='v4_reference':ms.extend(train_legacy(idx));ws.extend([w/3]*3)
  else:ms.append(train_new(name,idx,cache));ws.append(w)
 return ms,np.array(ws)


In [6]:
class NewSelector:
 def __init__(self,a,target,names):self.a=a;self.y=target;self.names=names;self.cache={}
 def select(self,method,k):
  if (method,k) in self.cache:return self.cache[(method,k)]
  numeric=np.array([s.startswith(('numeric:','module:')) for s in self.names])
  with warnings.catch_warnings():
   warnings.simplefilter('ignore',RuntimeWarning)
   global_score=np.nan_to_num(f_classif(self.a,self.y)[0],nan=0,posinf=0,neginf=0)
   if method=='balanced':
    orders=[]
    for c in range(NC):
     yes=self.y==c
     score=np.nan_to_num(chi2(self.a,yes)[0],nan=0,posinf=0,neginf=0)
     high=np.asarray(self.a[yes].mean(axis=0)).ravel()>np.asarray(self.a[~yes].mean(axis=0)).ravel()
     score[~high|numeric]=0
     orders.append(np.argsort(-score,kind='stable')[:min(k,int((score>0).sum()))])
    order=[]
    for rank in range(max(map(len,orders))):
     for o in orders:
      if rank<len(o):order.append(int(o[rank]))
    order+=list(np.argsort(-global_score,kind='stable'))
   else:order=list(np.argsort(-global_score,kind='stable'))
  z=self.a.tocsc();chosen=list(np.flatnonzero(numeric));seen=set(chosen);signatures=set()
  for j in order:
   j=int(j)
   if j in seen:continue
   seen.add(j)
   if global_score[j]<=0:continue
   lo,hi=z.indptr[j:j+2]
   sig=hashlib.sha256(z.indices[lo:hi].tobytes()+z.data[lo:hi].tobytes()).digest()
   if sig in signatures:continue
   signatures.add(sig);chosen.append(j)
   if len(chosen)>=k:break
  self.cache[(method,k)]=np.sort(chosen);return self.cache[(method,k)]

class SpecificityPost:
 def fit(self,a,names,target):
  self.names=np.array(names);self.binary=np.flatnonzero([not n.startswith(('numeric:','module:')) for n in names]);self.summary=np.flatnonzero([n.startswith(('numeric:','module:')) for n in names])
  z=(a[:,self.binary]!=0).astype(np.float32)
  counts=np.vstack([np.asarray(z[target==c].sum(axis=0)).ravel() for c in range(NC)])
  prior=np.bincount(target,minlength=NC)/len(target);freq=counts.sum(axis=0)
  posterior=(counts+10*prior[:,None])/(freq+10)
  kl=(posterior*np.log(np.maximum(posterior,1e-12)/prior[:,None])).sum(axis=0)
  self.weights=(.25+np.sqrt(np.maximum(kl,0))*freq/(freq+20)).astype(np.float32)
  self.scaler=MaxAbsScaler().fit(a[:,self.summary]);return self
 def transform(self,a):
  from sklearn.preprocessing import normalize
  block=normalize(a[:,self.binary].multiply(self.weights),norm='l2')
  return csr32(sparse.hstack([block,self.scaler.transform(a[:,self.summary])],format='csr'))

NEW={}
for view in ['raw','gated_syn_down_s3']:
 for method in ['global','balanced']:
  NEW[f'{view}_{method}_lgb4096']=dict(view=view,algo='lgb',k=4096,method=method)
for c in [.1,.5]:NEW[f'balanced_lr_C{c}']=dict(view='raw',algo='lr',k=2048,method='balanced',C=c)
for c in [1.,4.]:NEW[f'specificity_lr_C{c}']=dict(view='raw',algo='lr',k=4096,method='balanced',C=c,specificity=True)

def fit_new6(name,idx,cache):
 cfg=NEW[name];view=cfg['view']
 if view not in cache:
  enc,space,a,_=prepare_view(idx,view);cache[view]=(enc,space,a,NewSelector(a,yi[idx],space.names))
 enc,space,a,sel=cache[view];cols=sel.select(cfg['method'],cfg['k']);target=yi[idx]
 if cfg.get('specificity'):post=SpecificityPost().fit(a[:,cols],space.names[cols],target)
 else:post=ColumnPost().fit(a[:,cols],space.names[cols])
 est=estimator(cfg);tx=post.transform(a[:,cols])
 if cfg.get('augment'):
  if '_aug' not in cache:cache['_aug']=parse_events(rare_background(X.iloc[idx],[events[i] for i in idx],SEED+777,count=10))
  ar=enc.transform(cache['_aug'],exclude_events=[events[i] for i in idx]);ax=post.transform(space.transform(ar)[:,cols])
  tx=csr32(sparse.vstack([tx,ax],format='csr'));target=np.concatenate([target,target]);rw=np.r_[np.ones(len(idx)),np.full(len(idx),.5)]
 else:rw=np.ones(len(idx))
 if cfg['algo']=='xgb':est.fit(tx,target,sample_weight=compute_sample_weight('balanced',target)*rw)
 elif cfg.get('augment'):est.fit(tx,target,sample_weight=rw)
 else:est.fit(tx,target)
 m=Member(name,view,enc,space,cols,post,est)
 # Column order of a specificity block changes, but raw_names always describe the vectorizer input.
 if cfg.get('specificity'):m.names=np.concatenate([m.raw_names[post.binary],m.raw_names[post.summary]])
 return m
(OUT/'new_configurations.json').write_text(json.dumps(NEW,indent=2))
print('Loaded frozen v6 component definitions.')

NEW.update({'gated_s3_aug_global_lgb4096': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 4096, 'method': 'global', 'augment': True}, 'gated_s3_aug_balanced_lgb4096': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 4096, 'method': 'balanced', 'augment': True}, 'raw_aug_global_xgb4096_d3': {'view': 'raw', 'algo': 'xgb', 'k': 4096, 'method': 'global', 'depth': 3, 'augment': True}, 'raw_aug_global_xgb4096_d5': {'view': 'raw', 'algo': 'xgb', 'k': 4096, 'method': 'global', 'depth': 5, 'augment': True}})
(OUT/'new_configurations.json').write_text(json.dumps(NEW,indent=2))

print('Frozen v6 comparator ready.')


New configurations 8
Total new configurations including augmentation extension: 12


In [7]:
def fit_plan6(idx,plan,basefit=None):
 ms=[];ws=[];cache={}
 for name,w in plan:
  if name=='v5_reference':
   old,ow=basefit if basefit is not None else fit_base(idx)
   ms.extend(old);ws.extend(ow*w)
  else:ms.append(fit_new6(name,idx,cache));ws.append(w)
 return ms,np.array(ws)

V6_PLAN=[('v5_reference',.75),('gated_s3_aug_global_lgb4096',.25)]
def fit_v6(idx):return fit_plan6(idx,V6_PLAN)


In [8]:

# Optional acceleration: without these files the notebook refits v6 from scratch.
CACHE_V6=True
cache_v6_ready=False;cache_v6_validated=False;cached_v6_prob=None;cached_v6_bundle=None
v6_dir=WORK/'artifacts'/'v6'
if CACHE_V6 and all((v6_dir/f).exists() for f in ['oof_probabilities.npz','split_manifest.csv','selection.json']):
 saved=pd.read_csv(v6_dir/'split_manifest.csv',dtype={'group':str})
 assert saved.ID.equals(train.ID) and np.array_equal(saved.group.to_numpy(),groups.astype(str)), 'v6 cache patient/profile mismatch'
 sel6=json.loads((v6_dir/'selection.json').read_text())
 assert sel6['winner']=='v5plus_gated_s3_aug_global_lgb4096_0.25'
 q6=np.load(v6_dir/'oof_probabilities.npz',allow_pickle=True)
 assert np.array_equal(q6['dev'],dev) and np.array_equal(q6['labels'],labels)
 cached_v6_prob=.75*q6['v5_reference']+.25*q6['gated_s3_aug_global_lgb4096']
 cache_v6_ready=True
fit_v6_uncached=fit_v6

def fit_v6(idx):
 global cached_v6_bundle
 # Full-train cache is prohibited for CV and holdout fits.
 if CACHE_V6 and cache_v6_validated and np.array_equal(idx,np.arange(len(train))) and (v6_dir/'cancer_model_v6.pkl').is_file():
  if cached_v6_bundle is None:
   with open(v6_dir/'cancer_model_v6.pkl','rb') as f:cached_v6_bundle=pickle.load(f)
   assert cached_v6_bundle['version']==6 and list(cached_v6_bundle['genes'])==genes and np.array_equal(cached_v6_bundle['classes'],labels)
   assert cached_v6_bundle['plan']==[('v5_reference',.75),('gated_s3_aug_global_lgb4096',.25)]
  print('Reusing completed v6 full-train fit after selection; no full-train cache used in validation.',flush=True)
  return cached_v6_bundle['models'],np.asarray(cached_v6_bundle['weights'])
 return fit_v6_uncached(idx)


## 1. 가이드 주장의 실측 확인
APC 하나로 암종을 확정하거나 test의 COAD 개수를 가정하지 않습니다. 아래 비율은 P(암종|해당 표기 보유)로서 임상적 specificity(TN/(TN+FP))가 아닙니다. 정답 전체를 사용하는 이 기술통계는 후보 선택 점수가 아니라 설명 자료입니다. 실제 모델의 빈도/지문/선별은 fold 학습 안에서 다시 적합합니다.

In [9]:
audit_rows=[]
for g,cl in [('APC','COAD'),('VHL','KIRC'),('IDH1','LGG'),('BRAF','THCA'),('NPM1','LAML'),('TP53','OV')]:
 if g not in genes:continue
 presence=train[g].notna()&train[g].ne('WT');trunc=train[g].fillna('WT').map(lambda v:bool(re.search('fs|[*]',v)))
 for title,mask in [('any_mutation',presence),('stop_or_fs_not_all_deletions',trunc)]:
  audit_rows.append(dict(gene=g,feature=title,target=cl,all_carriers=int(mask.sum()),target_carriers=int((mask&train.SUBCLASS.eq(cl)).sum()),target_fraction_among_carriers=float(train.loc[mask,'SUBCLASS'].eq(cl).mean()) if mask.any() else 0.,target_mutation_rate=float(mask[train.SUBCLASS.eq(cl)].mean())))
audit=pd.DataFrame(audit_rows);audit.to_csv(OUT/'guide_claim_audit.csv',index=False);display(audit)


,gene,feature,target,all_carriers,target_carriers,target_fraction_among_carriers,target_mutation_rate
0,APC,any_mutation,COAD,367,160,0.435967,0.717489
1,APC,stop_or_fs_not_all_deletions,COAD,218,155,0.711009,0.695067
2,VHL,any_mutation,KIRC,326,151,0.463190,0.452096
3,VHL,stop_or_fs_not_all_deletions,KIRC,172,83,0.482558,0.248503
4,IDH1,any_mutation,LGG,441,180,0.408163,0.786026
5,IDH1,stop_or_fs_not_all_deletions,LGG,2,0,0.000000,0.000000
6,BRAF,any_mutation,THCA,438,186,0.424658,0.574074
7,BRAF,stop_or_fs_not_all_deletions,THCA,9,0,0.000000,0.000000
8,NPM1,any_mutation,LAML,56,41,0.732143,0.259494
9,NPM1,stop_or_fs_not_all_deletions,LAML,40,40,1.000000,0.253165


## 2. 새 변수와 학습 내 지문 인코딩
- 문서 유전자, 명시된 단백질 위치, stop/fs 표기, 동반 변이, 관측 WT와의 조합을 후보로 만듭니다. 성별·흡연·증폭·바이러스 정보는 원본에 없으므로 추정해 넣지 않습니다.
- BRAF V600 등 표지 × log(변이 유전자 수+1)를 추가해 같은 표지가 변이량에 따라 다르게 작용할 수 있게 합니다. 환자의 변이량을 지우거나 전체 행을 정규화하지 않습니다.
- 지문 점수는 각 학습에서 최소 5명 관측/해당 암종 최소 3명인 유전자의 양의 log-odds 차이를 관측 수/(관측 수+20)으로 축소하고, 암종별 상위 8개를 남깁니다. 합·변이량으로 나눈 값·표지 일치 수를 별도 변수로 둡니다.
- 라벨을 사용하는 지문 점수의 학습 입력은 **내부 3-fold 그룹 cross-fitting**으로 만듭니다. 자기 자신의 라벨이나 동일 프로필 그룹을 보지 않은 통계로 학습 입력을 생성하고, 외부 검증/test에는 외부 학습 전체로 적합한 인코더를 사용합니다. 증강된 학습 복사본에도 같은 내부 분할 통계를 사용합니다.
- 가이드 전용 작은 모델, 기존 변이 표현+가이드, 지문 점수 추가를 비교합니다. 가이드 변수도 학습 빈도 기준을 통과해야 하며 보호한 변수는 총 입력 예산 2,048/4,096개 안에 포함합니다.

In [10]:
GUIDE_GENES='APC TP53 KRAS FBXW7 PIK3CA VHL PBRM1 SETD2 BAP1 IDH1 IDH2 ATRX BRAF NPM1 DNMT3A FLT3 PTEN CTNNB1 RPL22 CTCF NRAS DSC1 DSC3 ACSM1 PCLO MXRA5 SYNE1 RYR2 KMT2D CDKN1A RXRA ARID1A KDM6A SPOP HRAS NF1 CHEK2 KIT CDH1 GATA3 MAP3K1 STK11 KEAP1 NFE2L2 SOX2 PKHD1 EGFR MET AXIN1 CDKN2A SMAD4 NOTCH1 LRIG1 THEM4 ERCC2 BTG2 SOCS1 DST PABPC1 MTOR XYLT2 MVK CD48 RB1'.split()
GUIDE_GENES=[g for g in GUIDE_GENES if g in genes]
HOTSPOTS={'IDH1':['R132'],'IDH2':['R140','R172'],'BRAF':['V600'],'VHL':['S111'],'NPM1':['W288'],'PTEN':['R130'],'SPOP':['F133','W131'],'HRAS':['Q61','G13','K117'],'NRAS':['Q61'],'KRAS':['G12','G13','Q61'],'KIT':['D816','N822'],'PIK3CA':['H1047','E545','E542'],'CHEK2':['K373'],'EGFR':['L858','E746']}
COMBOS=[('APC','TP53'),('APC','KRAS'),('APC','FBXW7'),('APC','PIK3CA'),('VHL','PBRM1'),('VHL','SETD2'),('VHL','BAP1'),('IDH1','TP53'),('IDH1','ATRX'),('IDH1','PTEN'),('IDH1','EGFR'),('PTEN','PIK3CA'),('PTEN','CTNNB1'),('PTEN','RPL22'),('PIK3CA','CDH1'),('PIK3CA','GATA3'),('PIK3CA','MAP3K1'),('KRAS','TP53'),('KRAS','CDKN2A'),('KRAS','SMAD4'),('KRAS','STK11'),('KRAS','KEAP1'),('TP53','CDKN1A'),('TP53','KMT2D'),('TP53','CDKN2A'),('TP53','NOTCH1'),('CTNNB1','AXIN1'),('NPM1','DNMT3A'),('NPM1','FLT3')]

def presence8(frame):return (frame[genes].notna()&frame[genes].ne('WT')).to_numpy(dtype=np.float32)

def guide_records8(frame,ev):
 wt=frame[genes].eq('WT').to_numpy();lookup={g:i for i,g in enumerate(genes)};out=[]
 for row,wrow in zip(ev,wt):
  by_gene=dict(row);mut=set(by_gene);burden=len(mut);logn=np.log1p(burden);d={'numeric:guide_log_burden':logn}
  for g in GUIDE_GENES:
   if g not in mut:continue
   d['guide_gene:'+g]=1.;d['guide_context:'+g+':log_burden']=logn
   if any(t.endswith('*') or 'fs' in t for t in by_gene[g]):d['guide_trunc:'+g]=1.
  for g,hs in HOTSPOTS.items():
   for h in hs:
    if any(re.match('^'+h+'([^0-9]|$)',t) for t in by_gene.get(g,())):
     d['guide_hot:'+g+':'+h]=1.;d['guide_context:'+g+':'+h+':log_burden']=logn
  for a,b in COMBOS:
   if a in mut and b in mut:d['guide_pair:'+a+':'+b]=1.
  for a,b in [('IDH1','PTEN'),('IDH1','EGFR'),('TP53','APC'),('BRAF','APC'),('KRAS','EGFR'),('TP53','PIK3CA')]:
   if a in mut and b in lookup and wrow[lookup[b]]:d['guide_observed_wt:'+a+':'+b]=1.
  if 'TP53' in mut and burden==1:d['guide_pattern:TP53_only_observed_mutated_gene']=1.
  if 'APC' in mut and any(t.endswith('*') or 'fs' in t for t in by_gene['APC']):d['guide_context:APC:trunc_log_burden']=logn
  out.append(d)
 return out

class Fingerprint8:
 def fit(self,P,target):
  freq=P.sum(axis=0);weights=[]
  for c in range(NC):
   yes=target==c;nc=int(yes.sum());positive=P[yes].sum(axis=0)
   p=(positive+1)/(nc+2);q=(freq-positive+1)/(len(target)-nc+2)
   score=np.maximum(0,np.log(p/(1-p))-np.log(q/(1-q)))*freq/(freq+20)
   score[(freq<5)|(positive<3)]=0
   chosen=np.argsort(-score,kind='stable')[:8];chosen=chosen[score[chosen]>0]
   weights.append((chosen,score[chosen].astype(np.float32)))
  self.markers=weights;return self
 def transform(self,P):
  out=np.zeros((len(P),NC*3),np.float32);burden=P.sum(axis=1)
  for c,(indices,w) in enumerate(self.markers):
   score=P[:,indices]@w;out[:,c*3]=score;out[:,c*3+1]=score/np.maximum(burden,1);out[:,c*3+2]=P[:,indices].sum(axis=1)
  return out
 def rows(self):
  return [dict(target=str(labels[c]),gene=genes[j],weight=float(w)) for c,(indices,ws) in enumerate(self.markers) for j,w in zip(indices,ws)]

def crossfit_fingerprint8(frame,target,profile_groups,augmented=None):
 P=presence8(frame);A=presence8(augmented) if augmented is not None else None
 out=np.zeros((len(frame),NC*3),np.float32);augout=np.zeros_like(out) if A is not None else None
 inner=StratifiedGroupKFold(3,shuffle=True,random_state=108).split(P,target,profile_groups)
 for a,b in inner:
  assert not set(profile_groups[a]).intersection(profile_groups[b])
  fp=Fingerprint8().fit(P[a],target[a]);out[b]=fp.transform(P[b])
  if A is not None:augout[b]=fp.transform(A[b])
 return out,augout

class Encoder8(EvidenceEncoder):
 def __init__(self,include_raw=True,use_fp=False):super().__init__(.25,.25,3);self.include_raw=include_raw;self.use_fp=use_fp;self.fp=None
 def records(self,frame,ev=None,training=False,exclude_events=None,fp_override=None,guide=None):
  if ev is None:ev=parse_events(frame[genes])
  rs=super().transform(ev,training=training,exclude_events=exclude_events) if self.include_raw else [{} for _ in ev]
  extra=guide_records8(frame,ev) if guide is None else guide
  fp=fp_override if fp_override is not None else (self.fp.transform(presence8(frame)) if self.use_fp else None)
  for i,(d,g) in enumerate(zip(rs,extra)):
   d.update(g)
   if fp is not None:
    for c in range(NC):
     for k,title in enumerate(['sum','mean_per_mutated_gene','matches']):d['fingerprint:'+str(labels[c])+':'+title]=float(fp[i,c*3+k])
  return rs

class Member8:
 def __init__(self,name,enc,space,cols,post,est,pair=None):
  self.name=name;self.view='v8_'+name;self.encoder=enc;self.vec=space.compact_vec(cols);self.raw_names=self.vec.get_feature_names_out();self.names=post.names;self.post=post;self.est=est;self.pair=pair
 def predict_records(self,rs):return self.est.predict_proba(self.post.transform(csr32(self.vec.transform(rs))))
 def predict_frame(self,frame):return self.predict_records(self.encoder.records(frame[genes]))

predict_members6=predict_members
def predict_members(members,weights,frame):
 old=[(m,w) for m,w in zip(members,weights) if not hasattr(m,'predict_frame')]
 p=predict_members6([m for m,w in old],[w for m,w in old],frame) if old else None
 for m,w in zip(members,weights):
  if hasattr(m,'predict_frame'):
   q=m.predict_frame(frame)*w;p=q if p is None else p+q
 return p

C8={
 'guide_only_lr_C0.1':dict(raw=False,fp=False,algo='lr',C=.1,k=1024,post='maxabs'),
 'guide_only_lr_C1':dict(raw=False,fp=False,algo='lr',C=1.,k=1024,post='maxabs'),
 'raw_guide_lr':dict(raw=True,fp=False,algo='lr',C=.1,k=2048,post='none'),
 'raw_guide_aug_lgb2048':dict(raw=True,fp=False,algo='lgb',k=2048,post='none',augment=True),
 'raw_guide_aug_lgb4096':dict(raw=True,fp=False,algo='lgb',k=4096,post='none',augment=True),
 'raw_guide_fingerprint_lr':dict(raw=True,fp=True,algo='lr',C=.1,k=2048,post='none'),
 'raw_guide_fingerprint_aug_lgb':dict(raw=True,fp=True,algo='lgb',k=2048,post='none',augment=True),
 'guide_fingerprint_only_lr':dict(raw=False,fp=True,algo='lr',C=1.,k=1024,post='maxabs')}

def select8(space,a,target,k):
 if a.shape[1]<=k:return np.arange(a.shape[1])
 selector=NewSelector(a,target,space.names)
 protected=np.flatnonzero([s.startswith(('guide_','fingerprint:','numeric:','module:')) for s in space.names])
 order=selector.select('global',min(a.shape[1],max(k*2,4096)))
 selected=list(protected[:k]);seen=set(selected)
 for j in order:
  if int(j) not in seen:selected.append(int(j));seen.add(int(j))
  if len(selected)>=k:break
 return np.sort(selected)

def prepare8(idx,raw,fp,cache):
 key=(raw,fp)
 if key in cache:return cache[key]
 if '_ev' not in cache:
  cache['_ev']=[events[i] for i in idx];cache['_guide']=guide_records8(X.iloc[idx],cache['_ev'])
  cache['_aug_frame']=rare_background(X.iloc[idx],cache['_ev'],SEED+777,count=10)
  cache['_aug_ev']=parse_events(cache['_aug_frame']);cache['_aug_guide']=guide_records8(cache['_aug_frame'],cache['_aug_ev'])
 enc=Encoder8(raw,fp).fit(cache['_ev']);fpo=fpa=None
 if fp:
  if '_fp_crossfit' not in cache:cache['_fp_crossfit']=crossfit_fingerprint8(X.iloc[idx],yi[idx],groups[idx],cache['_aug_frame'])
  fpo,fpa=cache['_fp_crossfit'];enc.fp=Fingerprint8().fit(presence8(X.iloc[idx]),yi[idx])
 rs=enc.records(X.iloc[idx],cache['_ev'],training=True,fp_override=fpo,guide=cache['_guide'])
 ar=enc.records(cache['_aug_frame'],cache['_aug_ev'],exclude_events=cache['_ev'],fp_override=fpa,guide=cache['_aug_guide'])
 space=Space();a=space.fit(rs);cache[key]=(enc,space,a,ar);return cache[key]

def train8(name,idx,cache):
 cfg=C8[name];enc,space,a,ar=prepare8(idx,cfg['raw'],cfg['fp'],cache);cols=select8(space,a,yi[idx],cfg['k'])
 post=ColumnPost(cfg['post']).fit(a[:,cols],space.names[cols]);tx=post.transform(a[:,cols]);target=yi[idx]
 est=estimator(cfg)
 if cfg.get('augment'):
  tx=csr32(sparse.vstack([tx,post.transform(space.transform(ar)[:,cols])],format='csr'));target=np.r_[target,target]
  est.fit(tx,target,sample_weight=np.r_[np.ones(len(idx)),np.full(len(idx),.5)])
 else:est.fit(tx,target)
 return Member8(name,enc,space,cols,post,est)

PAIRS8=[('GBMLGG','LGG'),('KIPAN','KIRC'),('LUAD','LUSC'),('THCA','SKCM')]
def train_pair8(pair,idx):
 selected=idx[np.isin(y[idx],pair)];target=(y[selected]==pair[1]).astype(int)
 ev=[events[i] for i in selected];enc=Encoder8(True,False).fit(ev);rs=enc.records(X.iloc[selected],ev,training=True)
 space=Space();a=space.fit(rs);cols=select8(space,a,target,1024);post=ColumnPost().fit(a[:,cols],space.names[cols])
 est=LogisticRegression(C=.1,class_weight='balanced',max_iter=2000).fit(post.transform(a[:,cols]),target)
 return Member8('pair_'+'_'.join(pair),enc,space,cols,post,est,pair=pair)

def refine_pair8(base,pair_probability,pair,alpha):
 out=base.copy();columns=np.searchsorted(labels,np.array(pair));mass=out[:,columns].sum(axis=1)
 old=out[:,columns]/np.maximum(mass[:,None],1e-12);mix=(1-alpha)*old+alpha*pair_probability
 out[:,columns]=mass[:,None]*mix;return out

assert not re.match('^V600([^0-9]|$)','V6000E') and re.match('^V600([^0-9]|$)','V600E')
(OUT/'configurations_v8.json').write_text(json.dumps(C8,indent=2));(OUT/'guide_features.json').write_text(json.dumps(dict(genes=GUIDE_GENES,hotspots=HOTSPOTS,pairs=COMBOS,specialists=PAIRS8),indent=2))
print('V8:',len(C8),'new models and',len(PAIRS8),'pair specialists')

# Controlled second stage: preserve the exact v6 component input instead of replacing it.
def canonical_column_signatures8(a):
 c=a.tocsc(copy=True);c.eliminate_zeros();c.sort_indices();result=[]
 for j in range(c.shape[1]):
  lo,hi=c.indptr[j:j+2];result.append(hashlib.sha256(c.indices[lo:hi].tobytes()+c.data[lo:hi].tobytes()).digest())
 return result

class PreservedMember8(Member8):
 def __init__(self,name,base_enc,base_space,base_cols,guide_enc,guide_space,guide_cols,est):
  self.name=name;self.view='v8_'+name;self.encoder=base_enc;self.guide_encoder=guide_enc;self.base_vec=base_space.compact_vec(base_cols);self.guide_vec=guide_space.compact_vec(guide_cols);self.est=est;self.pair=None
  self.raw_names=np.r_[self.base_vec.get_feature_names_out(),self.guide_vec.get_feature_names_out()];self.names=self.raw_names
 def predict_prepared_frame(self,frame,ev=None,guide=None):
  if ev is None:ev=parse_events(frame[genes])
  a=csr32(self.base_vec.transform(self.encoder.transform(ev)))
  b=csr32(self.guide_vec.transform(self.guide_encoder.records(frame,ev=ev,guide=guide)))
  return self.est.predict_proba(csr32(sparse.hstack([a,b],format='csr')))
 def predict_frame(self,frame):return self.predict_prepared_frame(frame[genes])

EXTRA8={
 'preserved_lr_plus_guide':dict(preserved=True,raw=True,fp=False,algo='lr',C=.1,k=2048,post='none'),
 'preserved_lr_plus_guide_fingerprint':dict(preserved=True,raw=True,fp=True,algo='lr',C=.1,k=2048,post='none'),
 'preserved_lgb_plus_guide':dict(preserved=True,raw=True,fp=False,algo='lgb',k=4096,post='none',augment=True),
 'preserved_lgb_plus_guide_fingerprint':dict(preserved=True,raw=True,fp=True,algo='lgb',k=4096,post='none',augment=True)}
C8.update(EXTRA8)
train8_first_stage=train8

def train8(name,idx,cache):
 cfg=C8[name]
 if not cfg.get('preserved'):return train8_first_stage(name,idx,cache)
 base_view='raw' if cfg['algo']=='lr' else 'gated_syn_down_s3';key=('preserved_base',base_view)
 if key not in cache:
  enc,space,a,sel=prepare_view(idx,base_view)
  cols=sel.select('chi2_full',.98,2048) if cfg['algo']=='lr' else NewSelector(a,yi[idx],space.names).select('global',4096)
  cache[key]=(enc,space,a,cols)
 enc,space,a,cols=cache[key]
 guide_enc,guide_space,guide_a,guide_ar=prepare8(idx,False,cfg['fp'],cache)
 base_signatures=set(canonical_column_signatures8(a[:,cols]))
 guide_cols=np.array([j for j,s in enumerate(canonical_column_signatures8(guide_a)) if s not in base_signatures],dtype=int)
 tx=csr32(sparse.hstack([a[:,cols],guide_a[:,guide_cols]],format='csr'));target=yi[idx]
 est=estimator(cfg)
 if cfg.get('augment'):
  ar=enc.transform(cache['_aug_ev'],exclude_events=cache['_ev'])
  ax=csr32(sparse.hstack([space.transform(ar)[:,cols],guide_space.transform(guide_ar)[:,guide_cols]],format='csr'))
  tx=csr32(sparse.vstack([tx,ax],format='csr'));target=np.r_[target,target]
  est.fit(tx,target,sample_weight=np.r_[np.ones(len(idx)),np.full(len(idx),.5)])
 else:est.fit(tx,target)
 return PreservedMember8(name,enc,space,cols,guide_enc,guide_space,guide_cols,est)
(OUT/'configurations_v8.json').write_text(json.dumps(C8,indent=2))
print('Additional four controlled settings preserve v6 component inputs. Total:',len(C8))


V8: 8 new models and 4 pair specialists


Four controlled additions preserve exact v6 component inputs; total 12 new models.


## 3. 같은 그룹 CV 비교
가이드의 감별 아이디어를 두 암종만 학습한 LR 전문가로도 비교합니다. 기존 두 암종에 배정된 확률 합을 보존하면서 내부 비율만 25%/50% 조정합니다. 암종 수나 비율을 강제로 맞추거나 규칙으로 정답을 덮어쓰지 않습니다. 뇌/신장만 또는 폐/갑상선-피부까지 포함하는 2개 범위를 비교합니다.

새 모델 단독 및 v6에 25%/50% 혼합을 비교하고, 기존 clean 0.6/희귀 변이 스트레스 0.3/중복 주석 0.1 기준을 유지합니다. 점수 +0.002, clean 하락 0.002 이하인 후보를 고정한 뒤 seed=71로 확인합니다. seed=71도 이전 v7에서 관측했으므로 독립 검증은 아닙니다.

In [11]:
P8={name:np.zeros((3,len(dev),NC),np.float32) for name in ['v6_reference',*C8]}
Q8={pair:np.zeros((3,len(dev),2),np.float32) for pair in PAIRS8};rows8=[]
for fold,(a,b) in enumerate(folds):
 tr=dev[a];va=dev[b];frames,evs,_=eval_views(va,tr,SEED+901+fold)
 if cache_v6_ready and cache_v6_validated:P8['v6_reference'][:,b]=cached_v6_prob[:,b]
 else:
  ms,ws=fit_v6(tr)
  for j,frame in enumerate(frames):P8['v6_reference'][j,b]=predict_members(ms,ws,frame)
  del ms
  if cache_v6_ready:
   assert np.allclose(P8['v6_reference'][:,b],cached_v6_prob[:,b],atol=1e-6);cache_v6_validated=True
   print('Fresh v6 first-fold probabilities match completed cache.',flush=True)
 cache={};guides=[guide_records8(f,ev) for f,ev in zip(frames,evs)];rec_cache={}
 for name,cfg in C8.items():
  start=time.time();m=train8(name,tr,cache)
  for j,frame in enumerate(frames):
   if hasattr(m,'base_vec'):P8[name][j,b]=m.predict_prepared_frame(frame,evs[j],guides[j])
   else:
    key=(cfg['raw'],cfg['fp'],j)
    if key not in rec_cache:rec_cache[key]=m.encoder.records(frame,evs[j],guide=guides[j])
    P8[name][j,b]=m.predict_records(rec_cache[key])
  assert np.allclose(P8[name][0,b],P8[name][2,b])
  print('V8',fold,name,'seconds',round(time.time()-start,1),flush=True);del m
 for pair in PAIRS8:
  m=train_pair8(pair,tr)
  for j,frame in enumerate(frames):Q8[pair][j,b]=m.predict_frame(frame)
  print('Pair',fold,pair,'done',flush=True);del m
 for name,p in P8.items():
  fs=[f1_score(yi[va],p[j,b].argmax(axis=1),average='macro',zero_division=0) for j in range(3)]
  rows8.append(dict(model=name,fold=fold,clean=fs[0],noise=fs[1],duplicate=fs[2],score=.6*fs[0]+.3*fs[1]+.1*fs[2]))
 pd.DataFrame(rows8).to_csv(OUT/'cv_folds.csv',index=False)
 np.savez_compressed(OUT/'oof_probabilities.npz',dev=dev,labels=labels,**P8)
 np.savez_compressed(OUT/'pair_oof.npz',**{'__'.join(k):v for k,v in Q8.items()})
 print('Completed CV fold',fold,flush=True)

def score8(p,splits):
 fs=np.array([[f1_score(yi[dev[b]],p[j,b].argmax(axis=1),average='macro',zero_division=0) for j in range(3)] for _,b in splits])
 return dict(clean=fs[:,0].mean(),noise=fs[:,1].mean(),duplicate=fs[:,2].mean(),score=(fs@np.array([.6,.3,.1])).mean())
PLANS8={'v6_reference':dict(kind='blend',members=[('v6_reference',1.)])}
for name in C8:
 PLANS8[name]=dict(kind='blend',members=[(name,1.)])
 for w in [.25,.5]:PLANS8[f'v6plus_{name}_{w}']=dict(kind='blend',members=[('v6_reference',1-w),(name,w)])
for scope in [2,4]:
 for alpha in [.25,.5]:PLANS8[f'pair_scope{scope}_a{alpha}']=dict(kind='pair',scope=scope,alpha=alpha)

def planned_oof8(plan):
 if plan['kind']=='blend':return sum(P8[name]*w for name,w in plan['members'])
 out=P8['v6_reference'].copy()
 for pair in PAIRS8[:plan['scope']]:
  for j in range(3):out[j]=refine_pair8(out[j],Q8[pair][j],pair,plan['alpha'])
 return out
rank8=pd.DataFrame({name:score8(planned_oof8(plan),folds) for name,plan in PLANS8.items()}).T.sort_values('score',ascending=False)
base=rank8.loc['v6_reference'];eligible=rank8[(rank8.clean>=base.clean-.002)&(rank8.score>=base.score+.002)]
winner8=eligible.index[0] if len(eligible) else 'v6_reference';plan8=PLANS8[winner8]
rank8.to_csv(OUT/'model_comparison.csv');(OUT/'selection.json').write_text(json.dumps(dict(winner=winner8,plan=plan8),indent=2))
print('Frozen candidate',winner8,flush=True);display(rank8.head(15))


Scoring completed eight-model stage plus four controlled additions and four pair specialists.
Frozen candidate v6_reference


,clean,noise,duplicate,score
v6plus_raw_guide_aug_lgb4096_0.25,0.432222,0.355806,0.432218,0.409297
pair_scope2_a0.25,0.436225,0.345792,0.434232,0.408896
pair_scope2_a0.5,0.436864,0.344399,0.434166,0.408855
v6plus_preserved_lgb_plus_guide_0.25,0.432440,0.352718,0.431826,0.408462
v6_reference,0.435576,0.345775,0.433301,0.408408
v6plus_preserved_lr_plus_guide_0.25,0.434684,0.347145,0.433526,0.408307
pair_scope4_a0.25,0.435577,0.344688,0.432483,0.408001
v6plus_preserved_lgb_plus_guide_fingerprint_0.25,0.433069,0.350438,0.430272,0.408000
v6plus_guide_only_lr_C0.1_0.25,0.434005,0.344925,0.432610,0.407142
v6plus_preserved_lgb_plus_guide_0.5,0.430225,0.351828,0.430672,0.406751


## 4. 고정 후보 재분할 점검
동일 개발 환자의 seed=71 분할이며 독립 새 환자는 아닙니다. 후보·비율을 조정하지 않고 점수 v6 이상, clean 하락 0.002 이하를 요구합니다. v6 예측은 모델별로 다시 계산하지 않고 같은 검증 프레임에서 한 번만 계산해 비교에 재사용합니다.

In [12]:
def fit_spec8(idx,spec,basefit=None):
 cache={};ms=[];ws=[];pair_models=[]
 if spec['kind']=='pair':
  ms,ws=basefit if basefit is not None else fit_v6(idx)
  pair_models=[train_pair8(pair,idx) for pair in PAIRS8[:spec['scope']]]
 else:
  for name,w in spec['members']:
   if name=='v6_reference':
    old,ow=basefit if basefit is not None else fit_v6(idx);ms.extend(old);ws.extend(ow*w)
   else:ms.append(train8(name,idx,cache));ws.append(w)
 return dict(kind=spec['kind'],models=ms,weights=np.asarray(ws),pairs=pair_models,alpha=spec.get('alpha'))

def predict_fitted8(fitted,frame,baseline=None):
 p=predict_members(fitted['models'],fitted['weights'],frame) if baseline is None else baseline.copy()
 if fitted['kind']=='pair':
  for m in fitted['pairs']:p=refine_pair8(p,m.predict_frame(frame),m.pair,fitted['alpha'])
 return p

confirm8=[];confirm_probs={name:np.zeros((3,len(dev),NC),np.float32) for name in ['v6_reference','v8_frozen']}
newfolds=list(StratifiedGroupKFold(3,shuffle=True,random_state=71).split(X.iloc[dev],y[dev],groups[dev]))
if winner8!='v6_reference':
 for fold,(a,b) in enumerate(newfolds):
  tr=dev[a];va=dev[b];frames,_,_=eval_views(va,tr,SEED+2901+fold);old=fit_v6(tr);candidate=fit_spec8(tr,plan8,old)
  for j,frame in enumerate(frames):
   bp=predict_members(old[0],old[1],frame);confirm_probs['v6_reference'][j,b]=bp
   if plan8['kind']=='pair':cp=predict_fitted8(candidate,frame,baseline=bp)
   else:
    cp=np.zeros_like(bp);new=[m for m in candidate['models'] if hasattr(m,'predict_frame')]
    for name,w in plan8['members']:
     if name=='v6_reference':cp+=bp*w
     else:cp+=next(m for m in new if m.name==name).predict_frame(frame)*w
   confirm_probs['v8_frozen'][j,b]=cp
  for title in confirm_probs:
   fs=[f1_score(yi[va],confirm_probs[title][j,b].argmax(axis=1),average='macro',zero_division=0) for j in range(3)]
   confirm8.append(dict(model=title,fold=fold,clean=fs[0],noise=fs[1],duplicate=fs[2],score=.6*fs[0]+.3*fs[1]+.1*fs[2]))
  pd.DataFrame(confirm8).to_csv(OUT/'confirmation_cv.csv',index=False);np.savez_compressed(OUT/'confirmation_probabilities.npz',dev=dev,labels=labels,**confirm_probs)
  print('Confirmation',fold,confirm8[-2:],flush=True);del old,candidate
 conf8=pd.DataFrame(confirm8).groupby('model')[['clean','noise','duplicate','score']].mean()
 accepted8=bool(conf8.loc['v8_frozen','score']>=conf8.loc['v6_reference','score'] and conf8.loc['v8_frozen','clean']>=conf8.loc['v6_reference','clean']-.002)
else:conf8=pd.DataFrame();accepted8=False
final_spec8=plan8 if accepted8 else PLANS8['v6_reference']
print('Accepted new model:',accepted8,flush=True);display(conf8)


Accepted new model: False


""


## 5. 홀드아웃 재현·전체 학습·피클·제출
홀드아웃은 이전에도 관측한 참고값입니다. 최종 후보 선택에는 다시 사용하지 않습니다. 미채택이면 v6를 유지하고 동일한 CSV를 새 제출로 배포하지 않습니다. 전체 train 캐시는 선택 후 전체 학습에서만 허용됩니다.

In [13]:
hf=fit_spec8(dev,final_spec8);hp=predict_fitted8(hf,X.iloc[hold]);pred=labels[hp.argmax(axis=1)]
metrics=dict(macro_f1=f1_score(y[hold],pred,average='macro',zero_division=0),accuracy=accuracy_score(y[hold],pred),weighted_f1=f1_score(y[hold],pred,average='weighted',zero_division=0),n=len(hold))
(OUT/'holdout_metrics.json').write_text(json.dumps(metrics,indent=2));np.savez_compressed(OUT/'holdout_probabilities.npz',probabilities=hp,indices=hold)
pd.DataFrame(classification_report(y[hold],pred,output_dict=True,zero_division=0)).T.to_csv(OUT/'holdout_classification_report.csv')
pd.DataFrame({'ID':train.ID.iloc[hold].values,'true':y[hold],'prediction':pred}).to_csv(OUT/'holdout_predictions.csv',index=False)
print('Previously observed holdout:',metrics,flush=True);del hf
fitted=fit_spec8(np.arange(len(train)),final_spec8)
flat=[(m,float(w),'blend') for m,w in zip(fitted['models'],fitted['weights'])]+[(m,None,'conditional_pair_refiner') for m in fitted['pairs']]
sizes=[];feature_rows=[];marker_rows=[]
for m,w,role in flat:
 sizes.append(dict(model=m.name,view=m.view,inputs=len(m.names),weight=w,role=role,pair=list(m.pair) if hasattr(m,'pair') and m.pair else None,pair_alpha=fitted['alpha'] if role=='conditional_pair_refiner' else None))
 for name in m.raw_names:feature_rows.append(dict(model=m.name,view=m.view,feature=name))
 if getattr(getattr(m,'guide_encoder',m.encoder),'fp',None) is not None:
  for row in getattr(m,'guide_encoder',m.encoder).fp.rows():marker_rows.append(dict(model=m.name,**row))
features=pd.DataFrame(feature_rows);features.to_csv(OUT/'final_features.csv',index=False);pd.DataFrame(marker_rows,columns=['model','target','gene','weight']).to_csv(OUT/'final_fingerprint_markers.csv',index=False)
(OUT/'model_sizes.json').write_text(json.dumps(sizes,indent=2))
def predict_bundle8(bundle,frame):return predict_fitted8(bundle['fitted'],frame)
bundle=dict(version=8,fitted=fitted,genes=genes,classes=labels,winner=winner8,plan=final_spec8,holdout_metrics=metrics,sizes=sizes,predict=predict_bundle8)
model_path=OUT/'cancer_model_v8.pkl'
with model_path.open('wb') as f:cloudpickle.dump(bundle,f,protocol=pickle.HIGHEST_PROTOCOL)
with model_path.open('rb') as f:loaded=pickle.load(f)
tp=loaded['predict'](loaded,test)
assert tp.shape==(len(test),NC) and np.isfinite(tp).all() and np.allclose(tp.sum(axis=1),1)
submission=sample.copy();submission.SUBCLASS=labels[tp.argmax(axis=1)];assert submission.ID.equals(test.ID)
submission.to_csv(OUT/'submission_v8.csv',index=False)
if accepted8:submission.to_csv(SUBMIT/'혜진_submission_v8.csv',index=False)
pd.DataFrame(tp,index=test.ID,columns=labels).to_csv(OUT/'test_probabilities.csv')
print('MODEL',model_path,'MiB',round(model_path.stat().st_size/1024**2,2),'new submission exported:',accepted8,flush=True)


Previously observed holdout: {'macro_f1': 0.4729888723709301, 'accuracy': 0.46333601933924257, 'weighted_f1': 0.46777775879200023, 'n': 1241}
Reusing completed v6 full-train fit after selection; no full-train cache used in validation.
MODEL /Users/mac/project/cancer_project/artifacts/v8/cancer_model_v8.pkl MiB 83.09 new submission exported: False


/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)


## 6. 독립 재로딩·문서·전체 변수 해석

In [14]:
import subprocess,tempfile,importlib.metadata
program="""
import sys,pickle,pandas as pd,numpy as np
with open(sys.argv[1],'rb') as f:b=pickle.load(f)
x=pd.read_csv(sys.argv[2]);s=pd.read_csv(sys.argv[3]);p=b['predict'](b,x)
assert s.ID.equals(x.ID) and np.array_equal(b['classes'][p.argmax(axis=1)],s.SUBCLASS.to_numpy())
assert np.isfinite(p).all() and np.allclose(p.sum(axis=1),1)
assert np.allclose(b['predict'](b,x.iloc[:8]),p[:8])
np.save(sys.argv[4],p)
"""
env=os.environ.copy();env.pop('PYTHONPATH',None)
with tempfile.TemporaryDirectory() as td:
 pfile=Path(td)/'prob.npy'
 r=subprocess.run([sys.executable,'-c',program,str(model_path.resolve()),str((DATA/'test.csv').resolve()),str((OUT/'submission_v8.csv').resolve()),str(pfile)],cwd=td,env=env,capture_output=True,text=True)
 if r.returncode:raise RuntimeError(r.stderr)
 assert np.allclose(np.load(pfile),tp)
proof=dict(status='passed',version=8,accepted_new_candidate=accepted8,winner=winner8,final_plan=final_spec8,primary=rank8.loc[winner8].to_dict(),baseline=rank8.loc['v6_reference'].to_dict(),confirmation=conf8.to_dict(orient='index'),holdout=metrics,full_independent_test_reload='passed',csv_exact_match=True,batch_invariance='passed',model_bytes=model_path.stat().st_size,public_score=None,v6_paired_cache_validated=cache_v6_validated,v6_full_fit_reused=cached_v6_bundle is not None,python=sys.version,data_sha256={f:hashlib.sha256((DATA/f).read_bytes()).hexdigest() for f in ['train.csv','test.csv','sample_submission.csv']},packages={k:importlib.metadata.version(k) for k in ['numpy','pandas','scipy','scikit-learn','lightgbm','xgboost','cloudpickle','matplotlib','nbformat','ipython','threadpoolctl']})
(OUT/'verification.json').write_text(json.dumps(proof,indent=2))
(WORK/'requirements-v8.txt').write_text('# Python '+platform.python_version()+'\n'+'\n'.join(k+'=='+v for k,v in proof['packages'].items())+'\n')

def markdown_table(frame):
 lines=['| '+' | '.join(map(str,frame.columns))+' |','| '+' | '.join(['---']*len(frame.columns))+' |']
 for row in frame.itertuples(index=False,name=None):lines.append('| '+' | '.join(f'{v:.5f}' if isinstance(v,(float,np.floating)) else str(v) for v in row)+' |')
 return '\n'.join(lines)

importance=[]
for m,w,role in flat:
 if hasattr(m.est,'feature_importances_'):values=m.est.feature_importances_;kind_name='tree_importance'
 elif hasattr(m.est,'coef_'):values=np.max(np.abs(m.est.coef_),axis=0);kind_name='max_absolute_coefficient'
 else:values=np.linalg.norm(m.est.coefs_[0],axis=1);kind_name='first_layer_weight_norm'
 for feature,value in zip(m.names,values):importance.append(dict(model=m.name,view=m.view,feature=feature,importance=float(value),importance_kind=kind_name,weight=w,role=role))
importance=pd.DataFrame(importance);importance.to_csv(OUT/'final_feature_importance.csv',index=False)
base_families={'gene','variant','position','type','aa','supported','missing'}
def related_genes8(feature):
 parts=feature.split(':')
 if parts[0] in base_families:return [parts[1]] if len(parts)>1 else []
 if parts[0].startswith('guide_'):return [s for s in parts[1:] if s in genes]
 return []
selected=importance.copy();selected['related_genes']=selected.feature.map(related_genes8)
expanded=selected.explode('related_genes');counts=expanded.groupby('related_genes').size();views=expanded.groupby('related_genes').view.agg(lambda a:', '.join(sorted(set(a))))
mut=(train[genes].notna()&train[genes].ne('WT'));te=(test[genes].notna()&test[genes].ne('WT'))
gdict=pd.DataFrame(dict(gene=genes,train_mutated=mut.sum().values,train_rate=mut.mean().values,test_rate=te.mean().values))
gdict['selected_model_feature_uses']=gdict.gene.map(counts).fillna(0).astype(int);gdict['selected_views']=gdict.gene.map(views).fillna('-')
gdict.to_csv(OUT/'gene_dictionary_v8.csv',index=False)
(WORK/'gene_dictionary_v8.md').write_text('# v8 전체 유전자 컬럼 사전\n\n관측 통계이며 임상적 기능/병원성 주석은 추정하지 않습니다. test_rate는 기술통계로만 사용했습니다. 조합 변수는 관련 유전자 각각에 집계하고 모델 간 중복도 포함합니다. 0이어도 숫자·경로 요약에 기여할 수 있습니다.\n\n'+markdown_table(gdict)+'\n')
meaning={'guide_gene':'가이드 유전자의 관측 변이 여부','guide_hot':'유전자·참조 아미노산·정확한 위치의 관측 표기','guide_trunc':'stop/fs 표기 여부; 기능 상실을 확정하지 않음','guide_pair':'두 유전자에서 함께 관측된 변이','guide_observed_wt':'첫 유전자 변이와 둘째 유전자의 관측 WT; 생물학적 정상 확정 아님','guide_context':'표지 존재와 log1p(변이 유전자 수)의 곱','guide_pattern':'관측 변이 패턴 조건; 암종 확정 규칙이 아님','fingerprint':'내부 그룹 cross-fitting한 암종별 유전자 연관 점수/변이량당 값/일치 수','gene':'유전자 변이의 가중 존재','variant':'정확한 정규화 변이 표기','position':'유전자와 단백질 위치','type':'유전자별 변이 유형','aa':'유전자와 아미노산 변화 종류','supported':'학습에서 지지된 비동의 변이 존재','numeric':'변이 부담 로그·유형 비율 등 수치 요약','module':'명시된 경로 유전자 목록의 가중 변이 요약','missing':'결측 표기'}
selected['meaning']=selected.feature.map(lambda s:meaning.get(s.split(':')[0],'노트북 정의 참조'));selected.to_csv(OUT/'final_feature_dictionary.csv',index=False)
slines=['# v8 최종 변수 전체 목록','','학습 내 빈도·선별과 CV 기준으로 선택한 실제 최종 입력입니다. 가이드·지문 변수는 학습 빈도 기준을 통과한 뒤 입력 예산 안에서 보호했습니다. 기존 v6 구성은 기존 선별을 유지합니다. 중요도는 모델 안에서만 비교하며 임상적 인과성/기여도를 증명하지 않습니다. 쌍별 전문가가 채택됐다면 weight는 선형 앙상블 가중치가 아니라 역할/alpha로 해석합니다.','','| model | view | feature | meaning | importance | metric |','| --- | --- | --- | --- | ---: | --- |']
for x in selected.itertuples(index=False):slines.append(f'| {x.model} | {x.view} | {x.feature} | {x.meaning} | {x.importance:.7g} | {x.importance_kind} |')
(WORK/'selected_variables_v8.md').write_text('\n'.join(slines)+'\n')

# Development error diagnostics compare the fixed candidate, even if it is rejected.
for title,p in [('v6',P8['v6_reference']),('v8_candidate',planned_oof8(plan8))]:
 target=yi[dev];predicted=p[0].argmax(axis=1)
 pd.DataFrame(classification_report(labels[target],labels[predicted],output_dict=True,zero_division=0)).T.to_csv(OUT/f'{title}_development_class_report.csv')
 cf=Counter(zip(labels[target[target!=predicted]],labels[predicted[target!=predicted]]))
 pd.DataFrame([(a,b,c) for (a,b),c in cf.most_common()],columns=['true','predicted','count']).to_csv(OUT/f'{title}_development_confusions.csv',index=False)
 burden=presence8(X.iloc[dev]).sum(axis=1);rows=[]
 for interval,lo,hi in [('1-10',1,11),('11-30',11,31),('31-100',31,101),('101+',101,10**9)]:
  mask=(burden>=lo)&(burden<hi)
  rows.append(dict(burden=interval,n=int(mask.sum()),present_classes=len(np.unique(target[mask])),accuracy=accuracy_score(target[mask],predicted[mask]),macro_f1_present_classes=f1_score(target[mask],predicted[mask],labels=np.unique(target[mask]),average='macro',zero_division=0)))
 pd.DataFrame(rows).to_csv(OUT/f'{title}_development_burden_report.csv',index=False)

comparison=pd.DataFrame(dict(v6=rank8.loc['v6_reference'],v8_candidate=rank8.loc[winner8])).reset_index(names='metric')
model_caption='채택 모델' if accepted8 else '유지 모델(v6; 신규 후보 평가 아님)'
report=f'''# 혜진 v8: Cancer Genome Guide 기반 실험

## 결과
초기 8개에 기존 구성 입력을 보존한 대조 4개를 추가한 신규 모델 12개와 쌍별 전문가 4개를 학습하고, 제한된 혼합·확률 재분배 계획을 같은 그룹 3-fold에서 비교했습니다. 1차 후보 `{winner8}`, 최종 채택 **{accepted8}**, 최종 구성 `{final_spec8}`입니다.

{markdown_table(comparison)}

{model_caption}의 기존 홀드아웃 Macro F1 {metrics['macro_f1']:.6f}, 정확도 {metrics['accuracy']:.6f}, weighted F1 {metrics['weighted_f1']:.6f}. 이 홀드아웃은 이미 여러 번 관측했으므로 독립 최종 성능이 아닙니다. v8 Public은 아직 확인하지 않았습니다.

## 문서 해석과 실측
가이드는 전체 train을 분석한 자료로, 유전자·핫스팟·조합 후보 설계에 참고했습니다. 전체 학습 정보를 본 문서에서 후보를 골랐으므로 설계 선택 편향이 있습니다. 검증 점수를 독립 새 데이터 성능으로 해석하지 않습니다.

{markdown_table(audit)}

APC stop/fs 표기 보유자 중 COAD 비율은 약 71.1%였으며, 가이드의 95% 이상이라는 확정 규칙을 사용하지 않았습니다. 위 비율은 P(암종|표기)로 임상적 specificity(TN/(TN+FP))가 아닙니다. test의 COAD 수가 570명이라는 주장도 정답으로 사용하지 않았습니다. 관측 변이 유전자 수는 mutations/Mb TMB가 아니며, 성별·흡연·CNV 증폭·바이러스 정보는 원본에 없어 추정하지 않았습니다.

## 새 변수와 근거
유전자 존재, 정확한 표지 위치, stop/fs, 동반 변이, 관측 WT 조합을 후보로 추가했습니다. BRAF V600 등 표지와 log1p(변이 유전자 수)의 곱은 표지의 맥락을 선형 모델도 표현하게 합니다. 단일 유전자나 변이량을 정답 규칙으로 쓰지 않습니다. stop/fs는 관측 표기이며 병원성·기능 상실을 확정하지 않습니다. 가이드 목록과 정확한 조합은 guide_features.json과 노트북에 포함했습니다.

암종별 지문 점수는 외부 학습의 유전자 빈도와 라벨로 계산한 양의 log-odds 차이입니다. 최소 관측 5명/해당 암종 3명, 빈도/(빈도+20) 축소, 암종별 상위 8개를 사용합니다. 학습 입력은 내부 그룹 3-fold에서 해당 그룹의 라벨을 제외한 cross-fitting으로 만들고, 증강 입력도 같은 내부 통계로 변환했습니다. 검증/test는 외부 학습 전체 통계를 사용합니다. 임상적 드라이버 판정이나 사전 확정 암종이 아닙니다.

초기 모델의 새 변수는 빈도 기준을 통과한 뒤 2,048/4,096개 예산 안에 포함해 보호했습니다. 추가 대조 4개는 기존 구성 모델의 선택 변수 2,048/4,096개를 그대로 유지하고 가이드/지문 변수만 덧붙였으며, 기준과 완전히 같은 열은 제외했습니다. 따라서 입력 수가 늘 수 있습니다. 초기 결과를 보고 추가한 적응 탐색이므로 선택 편향이 있습니다. 숫자·경로 요약을 보존하고 기존 변이량을 지우지 않았습니다. LightGBM 증강은 학습에서만 관측한 희귀 변이를 평균 10개 추가하고 원본/증강 가중치 1/0.5를 사용합니다. 증강이 라벨을 유지한다는 가정은 한계입니다.

## 쌍별 전문가
GBMLGG/LGG, KIPAN/KIRC, LUAD/LUSC, THCA/SKCM의 두 라벨만으로 각각 LR을 학습했습니다. 해당 두 암종의 기존 확률 합을 보존한 채 내부 비율만 alpha 0.25/0.5로 조정합니다. 대상 2쌍/4쌍을 비교했습니다. test의 암종 비율을 강제로 맞추거나 단서가 있다는 이유로 정답을 덮어쓰지 않습니다. 원본 프로필이 같아도 정답이 다른 사례는 이 방법만으로 모두 해결할 수 없습니다.

## 검증·선택
v6를 고정 대조군으로 사용하고 clean 0.6/희귀 관측 변이 스트레스 0.3/중복 주석 0.1을 유지했습니다. 1차 점수 +0.002 이상, clean 하락 0.002 이하인 후보를 고정한 뒤 seed=71에서 확인합니다. 재분할 점수가 v6 이상이며 clean 하락 0.002 이하여야 채택합니다. seed=71은 v7에서도 관측한 같은 개발 환자의 분할이며 독립 새 환자는 아닙니다. 여러 버전의 반복 탐색에 따른 선택 편향이 있습니다.

{markdown_table(conf8.reset_index()) if len(conf8) else '1차 기준을 넘는 후보가 없어 재분할 확인을 생략했습니다.'}

모든 적합·선별·통계·스케일러는 해당 학습에서만 계산했습니다. v6 캐시는 원본 ID/프로필/클래스·분할 확인과 첫 fold 실제 재학습 일치를 거친 완료 OOF만 재사용합니다. 전체 train 모델 캐시는 선택 후 최종 전체 학습에만 허용하며 CV·홀드아웃에는 금지합니다. 캐시가 없거나 CACHE_V6=False면 전부 재학습합니다. 원본 데이터 SHA256과 재현 환경을 verification.json에 기록했습니다.

## 후보별 결과
{markdown_table(rank8.reset_index(names='candidate'))}

## 모델·파일
cencer_v8.ipynb 하나에 모든 코드가 들어 있습니다. requirements-v8.txt 환경과 공유 data/가 필요합니다. 프로젝트 Python 파일이나 이전 모델 피클은 필수가 아닙니다. 모델 artifacts/v8/cancer_model_v8.pkl 크기는 {model_path.stat().st_size/1024**2:.2f} MiB, 분류기 입력 슬롯 합은 {sum(s['inputs'] for s in sizes):,}개로 고유 유전자 수와 다릅니다.

실제 변수·의미·중요도는 selected_variables_v8.md와 final_feature_dictionary.csv, 전체 유전자 관측 통계는 gene_dictionary_v8.md에 있습니다. 지문 점수가 최종 채택된 경우 실제 학습 표지는 final_fingerprint_markers.csv에 기록했습니다. 중요도 크기는 다른 모델 사이에서 직접 비교할 수 없고 인과적 기여도가 아닙니다.

피클은 프로젝트 코드 없는 임시 폴더에서 로딩해 test 2,546개의 CSV·ID·확률·배치 일치를 확인했습니다. 미채택 시 주 모델과 제출은 v6로 유지하며 artifacts/v8의 피클/CSV는 재현 확인용입니다. 동일 예측을 새 제출로 배포하지 않습니다. artifacts는 기존 방침대로 Git에서 제외합니다.
'''
report+='\n## 주요 오분류 조합(v6 개발 OOF)\n\n'+markdown_table(pd.read_csv(OUT/'v6_development_confusions.csv').head(10))+'\n'
report+='\n## 변이량별 정확도(v6 개발 OOF)\n\n'+markdown_table(pd.read_csv(OUT/'v6_development_burden_report.csv'))+'\n구간별 환자·암종 구성이 달라 구간 간 차이를 변이량의 인과효과로 해석하지 않습니다.\n'
(WORK/'analysis_report_v8.md').write_text(report)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
show=rank8.head(7)
if 'v6_reference' not in show.index:show=pd.concat([show,rank8.loc[['v6_reference']]])
fig,ax=plt.subplots(figsize=(12,6));xx=np.arange(len(show));ax.barh(xx-.18,show.clean,height=.35,label='Clean macro F1');ax.barh(xx+.18,show.noise,height=.35,label='Rare-variant stress macro F1');ax.set_yticks(xx,show.index);ax.invert_yaxis();ax.set_xlabel('Mean group 3-fold macro F1');ax.legend();fig.tight_layout();fig.savefig(OUT/'validation_comparison_v8.png',dpi=150);plt.close(fig)
p=WORK/'analysis_report_v8.md';p.write_text(p.read_text()+'\n![Development CV](artifacts/v8/validation_comparison_v8.png)\n')
print('Independent replay, reports, full feature dictionaries and diagnostics passed.',flush=True)


Independent replay, reports, full feature dictionaries and diagnostics passed.


Matplotlib is building the font cache; this may take a moment.


## 별도 실험용 후보
주 모델 채택과 구분해 약한 개발 CV 개선만 있는 후보를 별도 파일로 기록합니다. 재분할·홀드아웃·Public 개선을 확인한 모델이 아닙니다. 자동 제출하지 않습니다.

In [15]:
# Explicitly experimental output: never replaces the accepted primary model.
experimental_info=None
if not accepted8:
 base_metrics=rank8.loc['v6_reference']
 options=rank8[(rank8.index!='v6_reference')&(rank8.clean>base_metrics.clean)&(rank8.score>=base_metrics.score)].sort_values('clean',ascending=False)
 if len(options):
  experimental_name=str(options.index[0]);experimental_spec=PLANS8[experimental_name]
  experimental_fitted=fit_spec8(np.arange(len(train)),experimental_spec,basefit=(fitted['models'],fitted['weights']))
  experimental_prob=predict_fitted8(experimental_fitted,test)
  assert np.isfinite(experimental_prob).all() and np.allclose(experimental_prob.sum(axis=1),1)
  experimental_csv=sample.copy();experimental_csv.SUBCLASS=labels[experimental_prob.argmax(axis=1)]
  experimental_path=OUT/'cancer_model_v8_experimental.pkl'
  experimental_bundle=dict(version=8,experimental=True,fitted=experimental_fitted,genes=genes,classes=labels,winner=experimental_name,plan=experimental_spec,development_cv=rank8.loc[experimental_name].to_dict(),holdout_metrics=None,predict=predict_bundle8)
  with experimental_path.open('wb') as f:cloudpickle.dump(experimental_bundle,f,protocol=pickle.HIGHEST_PROTOCOL)
  csv_path=SUBMIT/'혜진_submission_v8_experimental.csv';experimental_csv.to_csv(csv_path,index=False);experimental_csv.to_csv(OUT/'submission_v8_experimental.csv',index=False)
  env=os.environ.copy();env.pop('PYTHONPATH',None)
  program_experiment="""
import sys,pickle,pandas as pd,numpy as np
with open(sys.argv[1],'rb') as f:b=pickle.load(f)
x=pd.read_csv(sys.argv[2]);s=pd.read_csv(sys.argv[3]);p=b['predict'](b,x)
assert s.ID.equals(x.ID) and np.array_equal(b['classes'][p.argmax(axis=1)],s.SUBCLASS.to_numpy())
assert np.allclose(b['predict'](b,x.iloc[:8]),p[:8])
np.save(sys.argv[4],p)
"""
  with tempfile.TemporaryDirectory() as td:
   pf=Path(td)/'p.npy'
   rr=subprocess.run([sys.executable,'-c',program_experiment,str(experimental_path.resolve()),str((DATA/'test.csv').resolve()),str(csv_path.resolve()),str(pf)],cwd=td,env=env,capture_output=True,text=True)
   if rr.returncode:raise RuntimeError(rr.stderr)
   assert np.allclose(np.load(pf),experimental_prob)
  changed=int((experimental_csv.SUBCLASS!=submission.SUBCLASS).sum())
  experimental_info=dict(status='passed',experimental=True,candidate=experimental_name,plan=experimental_spec,development_cv=rank8.loc[experimental_name].to_dict(),baseline_cv=base_metrics.to_dict(),accepted_primary=False,confirmation_evaluated=False,holdout_evaluated=False,public_score=None,changed_predictions_vs_v6=changed,model_bytes=experimental_path.stat().st_size,csv_exact_match=True,batch_invariance=True)
  (OUT/'experimental_verification.json').write_text(json.dumps(experimental_info,indent=2))
  rows=[]
  for m in experimental_fitted['pairs']:
   for j,name in enumerate(m.raw_names):rows.append(dict(model=m.name,pair=' / '.join(m.pair),feature=name,coefficient=float(m.est.coef_[0,j]),alpha=float(experimental_fitted['alpha'])))
  pair_features=pd.DataFrame(rows);pair_features.to_csv(OUT/'experimental_pair_features.csv',index=False)
  (OUT/'experimental_test_probabilities.csv').write_text(pd.DataFrame(experimental_prob,index=test.ID,columns=labels).to_csv())
  add='\n## 미채택 실험용 후보\n\n'
  add+=f'주 모델은 v6를 유지합니다. `{experimental_name}`는 메인 채택 기준을 넘지 못한 **실험용** 후보입니다. 개발 CV에서 원본 F1과 종합 점수가 v6보다 높았던 조합 중 원본 F1이 가장 높은 것을 기록했습니다. 재분할·새 홀드아웃·Public 개선은 확인하지 않았습니다. 안정적인 성능 향상이나 메인 모델 채택을 뜻하지 않습니다.\n\n'
  add+=markdown_table(pd.DataFrame(dict(v6=base_metrics,experimental=rank8.loc[experimental_name])).reset_index(names='metric'))
  add+=f'\n\n구성 `{experimental_spec}`. 파일 `submission/혜진_submission_v8_experimental.csv`, 모델 `artifacts/v8/cancer_model_v8_experimental.pkl`입니다. 테스트 예측 변경은 {changed}개이며 정답 증가 수가 아닙니다. 독립 폴더에서 전체 test CSV·ID·배치 일치를 검사했습니다. 기본 v8 CSV는 v6 재현용이고, 실제로 다른 답안은 experimental 파일입니다. 실험용 모델의 새 변수는 experimental_pair_features.csv에 있습니다.\n'
  path=WORK/'analysis_report_v8.md';body=path.read_text();marker='\n## 미채택 실험용 후보\n';path.write_text(body.split(marker)[0]+add)
  if len(pair_features):
   path=WORK/'selected_variables_v8.md';body=path.read_text();marker='\n## 실험용 쌍별 전문가 추가 변수\n';path.write_text(body.split(marker)[0]+marker+'\n아래는 미채택 실험용 모델에만 추가된 두 암종 분류기의 입력입니다. 기존 v6 입력은 위 목록과 같고 alpha는 확률 내부 비율 혼합값입니다. 계수 부호는 pair 두 번째 암종의 log-odds 방향이며 인과효과가 아닙니다.\n\n'+markdown_table(pair_features)+'\n')
  print('EXPERIMENTAL CSV:',csv_path,'candidate:',experimental_name,'changed:',changed,flush=True)
 else:print('No weak-improvement experimental candidate found; no experimental CSV generated.')

if experimental_info is not None and experimental_spec['kind']=='pair':
 reference_prob=pd.read_csv(OUT/'test_probabilities.csv',index_col=0)
 experimental_saved=pd.read_csv(OUT/'experimental_test_probabilities.csv',index_col=0)
 assert reference_prob.index.equals(experimental_saved.index) and reference_prob.columns.equals(experimental_saved.columns)
 pair_columns=[c for pair in PAIRS8[:experimental_spec['scope']] for c in pair]
 remaining_columns=[c for c in reference_prob if c not in pair_columns]
 assert np.allclose(reference_prob[remaining_columns],experimental_saved[remaining_columns])
 for pair in PAIRS8[:experimental_spec['scope']]:assert np.allclose(reference_prob[list(pair)].sum(axis=1),experimental_saved[list(pair)].sum(axis=1))
 experimental_info['pair_probability_mass_preserved']=True
 experimental_info['other_class_probabilities_unchanged']=True
 (OUT/'experimental_verification.json').write_text(json.dumps(experimental_info,indent=2))
 print('Experimental pair probability mass and other class probabilities verified.')


EXPERIMENTAL CSV: /Users/mac/project/cancer_project/submission/혜진_submission_v8_experimental.csv candidate: pair_scope2_a0.5 changed: 40


/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0 0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)


Experimental pair probability mass and other class probabilities verified.
